# 05 · 인간의 검증력: 시각화로 오류 잡기  (모듈 5)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/05_human_verification.ipynb)

> AI가 만든 "그럴듯한 분석"을 **인간의 3대 무기 — 🖼️시각화 · 🧭도메인 지식 · 👥동료 회람**으로 검증한다.
> AI는 빠르지만 *보지 못하고, 맥락이 없고, 혼자* 일한다.

In [ ]:
# [한 번 실행하고 넘어가세요] 한글 폰트 설정 — Colab 그래프의 한글이 깨지지 않도록(처음 1회 ~20초)
import os, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"   # 나눔고딕 폰트 경로
    if not os.path.exists(p):                                # 없으면
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")  # 설치
    fm.fontManager.addfont(p)                               # matplotlib에 폰트 등록
    plt.rc("font", family="NanumGothic")                   # 기본 글꼴로 지정
except Exception:
    pass                                                    # 실패해도 그냥 진행(영문은 정상)
plt.rc("axes", unicode_minus=False)                         # 마이너스 기호 깨짐 방지

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt   # plt = 그래프 그리기
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/wdi_panel.csv").dropna(subset=["gdp_pc","life_exp"]).copy()
df["log_gdp"] = np.log(df["gdp_pc"])

## 시나리오 — "소득이 높을수록 기대수명이 길다"
AI에게 분석을 시켰더니 상관계수를 줬다. **숫자만 보면 완벽해 보인다.**

In [ ]:
# 상관계수 = 두 변수가 함께 움직이는 정도(-1~1, 1에 가까울수록 강한 양의 관계)
print("상관계수(log소득 vs 기대수명):", round(df["log_gdp"].corr(df["life_exp"]), 3))

### 🖼️ 무기 1 — **그려본다**
현업 데이터엔 입력 오류가 섞이기 쉽다. 한 나라의 1인당 GDP가 **단위 실수로 1,000배** 잘못
입력됐다고 하자(흔한 오류). 숫자로는 안 보이지만 **그리면** 바로 드러난다.

In [ ]:
dirty = df.copy()                                              # 오류를 심을 복사본
idx = dirty.index[dirty["economy"].eq("VNM") & dirty["year"].eq(2021)]  # 베트남 2021년 행의 위치
dirty.loc[idx, "gdp_pc"] = dirty.loc[idx, "gdp_pc"] * 1000      # 단위 실수 모사: GDP를 1,000배로
dirty["log_gdp"] = np.log(dirty["gdp_pc"])                      # 로그 다시 계산

fig, ax = plt.subplots(1, 2, figsize=(11, 4))                  # 그래프 2개를 가로로 나란히
# 산점도(scatter): 점 하나 = 한 나라-연도. s = 점 크기
ax[0].scatter(dirty["log_gdp"], dirty["life_exp"], s=8); ax[0].set_title("AI가 받은 데이터(오류 포함)")
ax[1].scatter(df["log_gdp"], df["life_exp"], s=8, c="green"); ax[1].set_title("정상")
for a in ax: a.set_xlabel("log(1인당 GDP)"); a.set_ylabel("기대수명")  # 두 그래프 축 라벨
plt.tight_layout(); plt.show()                                 # 여백 정리 후 화면에 표시

→ 왼쪽 그림에 **오른쪽으로 멀리 튄 점**이 한눈에. 표의 숫자만 봤다면 회귀가 왜곡됐을 것이다.

### 🖼️ 같은 오류, 여러 그림 — "각도를 바꾸면 보인다"
산점도에서 튄 그 점은 **히스토그램·박스플롯에서도** 똑같이 드러난다. 한 각도로 안 보여도 다른 그림이 잡는다.

In [ ]:
# 같은 오류를 '다른 그림'으로도 — 산점도 말고 히스토그램·박스플롯에서도 똑같이 튄다
fig, ax = plt.subplots(1, 2, figsize=(11, 4))                  # 그래프 2개를 가로로
# (왼쪽) 히스토그램 — 오류값은 단 1건(막대 높이 1)이라, 세로축을 로그로 해야 그 외딴 막대가 보인다
ax[0].hist(dirty["log_gdp"], bins=40, color="#e45756", edgecolor="white")
ax[0].set_yscale("log")                                        # 높이 1짜리 외딴 막대도 보이도록 로그 눈금
err = dirty["log_gdp"].max()                                   # 오류값(비정상적으로 큰 log소득)
ax[0].annotate("오류값", xy=(err, 1), xytext=(err - 3.5, 50),  # 그 외딴 막대를 화살표로 가리킴
               arrowprops=dict(arrowstyle="->"), ha="center")
ax[0].set_xlabel("log(1인당 GDP)"); ax[0].set_ylabel("국가-연도 수 (로그 눈금)")
ax[0].set_title("히스토그램 — 오른쪽 끝 외딴 막대")
# (오른쪽) 박스플롯 — 수염(정상 범위) 밖으로 동떨어진 점 하나(이상치)
ax[1].boxplot(dirty["log_gdp"].dropna(), showmeans=True)
ax[1].set_xticks([1]); ax[1].set_xticklabels(["전체"]); ax[1].set_ylabel("log(1인당 GDP)")
ax[1].set_title("박스플롯 — 수염 밖 유일한 이상치")
plt.tight_layout(); plt.show()

In [ ]:
# 범인 찾기: GDP가 높은 순으로 정렬해 비정상적으로 부유해 보이는 나라-연도 확인
sus = dirty.sort_values("gdp_pc", ascending=False).head(3)   # 내림차순 상위 3개
sus[["name","year","gdp_pc","life_exp"]]

### 🧭 무기 2 — **도메인 지식**
"베트남 1인당 GDP가 한 해에 갑자기 세계 최고? 현장을 아는 사람은 *말이 안 된다*는 걸 안다."
AI엔 현장의 사실(ground truth)이 없다. **여러분에겐 있다.**

### 👥 무기 3 — **동료 회람**
1장 요약을 그 지역을 아는 **동료에게 회람**: "이 수치 단위가 이상한데요?"
한 사람(혹은 AI)이 놓친 걸 **여러 눈**이 잡는다.

---
✅ **결론**: AI가 분석을 *쉽게* 만들어도, *맞게* 만드는 건 인간이다.
시각화·도메인·회람 — 이 무기를 매번 쓰자. (체크리스트 → `handouts/verification_checklist.md`)